In [0]:
%python

from datetime import datetime, timezone
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType

startTime = datetime.now(timezone.utc)
dbutils.widgets.text("catalogName", 'mde_dev')
dbutils.widgets.text("deltaSchemaName", 'gold')
dbutils.widgets.text("deltaTableName", 'dim_manager')
dbutils.widgets.text("transformId", '-1') # TransformId from statedb

In [0]:
%python
pCatalogName = dbutils.widgets.get("catalogName")
pDimSchemaName = dbutils.widgets.get("deltaSchemaName")
pDimTableName = dbutils.widgets.get("deltaTableName")

pInputSchemaName = 'gold'
pInputTableOrViewName = f"v_{pDimTableName}"

pDimContainer = 'gold' #dbutils.widgets.get("deltaContainer")
pDimFolderPath = 'databricks/' #dbutils.widgets.get("deltaFolderPath")
pTransformId = int(dbutils.widgets.get("transformId")) # TransformId from statedb

In [0]:
%run ../../LakehouseUtilities

In [0]:
%python
spark.catalog.setCurrentCatalog(pCatalogName)

In [0]:
%python
spark.sql("""
    CREATE OR REPLACE TABLE {pCatalogName}.{pInputSchemaName}.{pInputTableOrViewName} AS
    SELECT
        p.ProductID,
        p.Name,
        p.ProductNumber,
        p.MakeFlag,
        p.FinishedGoodsFlag,
        p.Color,
        p.SafetyStockLevel,
        p.ReorderPoint,
        p.StandardCost,
        p.ListPrice,
        p.Size,
        p.SizeUnitMeasureCode,
        p.WeightUnitMeasureCode,
        p.Weight,
        p.DaysToManufacture,
        p.ProductLine,
        p.Class,
        p.Style,
        p.ProductSubcategoryID,
        p.ProductModelID,
        p.SellStartDate,
        p.SellEndDate,
        p.DiscontinuedDate,
        c.ProductCategoryID,
        c.Name AS CategoryName,
        sop.SpecialOfferID,
        so.Description AS SpecialOfferDescription,
        so.DiscountPct,
        so.Type AS SpecialOfferType,
        so.Category AS SpecialOfferCategory,
        so.StartDate AS SpecialOfferStartDate,
        so.EndDate AS SpecialOfferEndDate
    FROM
        mde_dev.bronze.product p
    LEFT JOIN
        mde_dev.bronze.productcategory c
    ON
        p.ProductSubcategoryID = c.ProductCategoryID
    LEFT JOIN
        mde_dev.bronze.specialofferproduct sop
    ON
        p.ProductID = sop.ProductID
    LEFT JOIN
        mde_dev.bronze.specialoffer so
    ON
        sop.SpecialOfferID = so.SpecialOfferID
""")

In [0]:
%python
dim = DimensionProcessor(
    schema_name=pDimSchemaName,
    table_name=pDimTableName,
    input_schema_name=pInputSchemaName,
    input_view_or_table_name=pInputTableOrViewName,
    delta_path=f"abfss://{pDimContainer}@{pStorageAccountEndpoint.replace('https://','')}{pDimFolderPath}{pDimTableName}",
    transform_id=pTransformId,
    catalog_name=pCatalogName
)

In [0]:
%python
try:
    dim.process()
    insert_default_row_sql(pDimSchemaName, pDimTableName, pTransformId)
except Exception as e:
    LakeOrchestrator.exit_with_status(success=False, startTime=startTime, message=str(e))

In [0]:
%python
LakeOrchestrator.exit_with_status(success=True, startTime=startTime)